# 1 read in cellbender filtered matrices

In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/IX_snMOseq_analysis/CellRangerARC-2.0.2_count_mergedPeaks/cellbender_out/'
MAL004_path = '/staging/leuven/stg_00171/Mark/IX_Ovul_9_Pcan_CR/cellbender_out/MAL004_reseq2_cb/'

# save as h5ad files for scanpy
IX_1 = sc.read_h5ad(filename=work_dir + 'adata_IX_1_cb.h5ad') # multiome RNA
IX_2 = sc.read_h5ad(filename=work_dir + 'adata_IX_2_cb.h5ad') # multiome RNA
IX_3 = sc.read_h5ad(filename=MAL004_path + 'adata_IX_3_cb.h5ad') # GEM-X RNA


In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/XII_snMOseq_2022-12-13/CellRangerARC-2.0.2_count_mergedPeaks/cellbender_outs/scanpy/'

XII_1 = sc.read_h5ad(filename=work_dir + 'adata_XII_1_cb.h5ad') # multiome RNA
XII_2 = sc.read_h5ad(filename=work_dir + 'adata_XII_2_cb.h5ad') # multiome RNA
XII_3 = sc.read_h5ad(filename=work_dir + 'adata_XII_3_cb.h5ad') # multiome RNA

In [ ]:
base_dir = '/staging/leuven/stg_00171/Mark/XV_snRNAseq_250424/cellbender_mergedPeaks/'

XV_1 = sc.read_h5ad(filename=base_dir + 'adata_XV_1_cb.h5ad') # GEM-X RNA
XV_2 = sc.read_h5ad(filename=base_dir + 'adata_XV_2_cb.h5ad') # GEM-X RNA


In [ ]:
adata_list = [IX_1, IX_2, IX_3, XII_1, XII_2, XII_3, XV_1, XV_2]

# 2 scanpy preprocessing

In [ ]:
for adata in adata_list:
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    print(adata)

In [ ]:
# mitchondrial genes
for adata in adata_list:
    adata.var["mt"] = adata.var_names.str.startswith("nbisL1-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

In [ ]:
adata_sc_list = []
for adata in adata_list:
    adata = adata[adata.obs.n_genes_by_counts < 10000, :]
    adata = adata[adata.obs.total_counts < 20000, :]
    adata = adata[adata.obs.pct_counts_mt < 0.5, :].copy()
    adata_sc_list.append(adata)
    print(adata)

In [ ]:
var_names = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']

for adata in adata_sc_list:
    sc.pl.violin(adata, var_names, jitter=0.4)

In [ ]:
# Normalize and scale
adata_sc_list_log = []

for adata in adata_sc_list:
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)
    adata_sc_list_log.append(adata)

In [ ]:
adata_sc_list_log_raw = []

for adata in adata_sc_list_log:
    adata.raw = adata.copy()
    adata_sc_list_log_raw.append(adata)

# 3 doublet detection and filtering

In [ ]:
import scrublet as scr
import scipy

In [ ]:
# run scrublet with predetermined threshold 0.5

adata_list_scrub = []

for adata in adata_sc_list_log_raw:
    # If adata.X is sparse and you need a dense matrix
    counts_matrix = adata.raw.X if adata.raw is not None else adata.X
    counts_matrix = counts_matrix.toarray() if scipy.sparse.issparse(counts_matrix) else counts_matrix
    scrub = scr.Scrublet(counts_matrix)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30)
    fixed_doublets = scrub.call_doublets(threshold=0.5)
    scrub.plot_histogram();
    adata.obs['doublet_score'] = doublet_scores
    adata.obs['predicted_doublet'] = predicted_doublets
    adata.obs['fixed_doublet'] = fixed_doublets
    adata = adata[~adata.obs['fixed_doublet']]
    adata_list_scrub.append(adata)

In [ ]:
for adata in adata_list_scrub:
    print(adata)

# 4 merge adata objects

In [ ]:
IX_1 = adata_list_scrub[0]
IX_2 = adata_list_scrub[1]
IX_3 = adata_list_scrub[2]
XII_1 = adata_list_scrub[3]
XII_2 = adata_list_scrub[4]
XII_3 = adata_list_scrub[5]
XV_1 = adata_list_scrub[6]
XV_2 = adata_list_scrub[7]

In [ ]:
IX = IX_1.concatenate([IX_2,IX_3], batch_categories=["IX_1", "IX_2", "IX_3"])
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
IX.write_h5ad(work_dir + 'IX_cb_sc_scr.h5ad')

In [ ]:
XII = XII_1.concatenate([XII_2, XII_3], batch_categories=["XII_1", "XII_2", "XII_3"])
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
XII.write_h5ad(work_dir + 'XII_cb_sc_scr.h5ad')

In [ ]:
XV = XV_1.concatenate(XV_2, batch_categories=["XV_1", "XV_2"])
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
XV.write_h5ad(work_dir + 'XV_cb_sc_scr.h5ad')

# 5 subset to same amount of cells per stage for integration

In [ ]:
div = len(XII.obs)  

ix = len(IX.obs)/div
xii = len(XII.obs)/div 
xv = len(XV.obs)/div

In [ ]:
import numpy as np

# Random reproducible seed
np.random.seed(33)  # Annas lucky number

In [ ]:
cell_indices = np.arange(IX.n_obs)
subset_indices = np.random.choice(cell_indices, size= int(IX.n_obs // ix)+1, replace=False)
print(len(cell_indices))
print(len(subset_indices))

In [ ]:
# Subset the AnnData object
IX_subset = IX[subset_indices, :]
IX_subset

In [ ]:
cell_indices = np.arange(XII.n_obs)
subset_indices = np.random.choice(cell_indices, size= int(XII.n_obs // xii), replace=False)
print(len(cell_indices))
print(len(subset_indices))

In [ ]:
# Subset the AnnData object
XII_subset = XII[subset_indices, :]
XII_subset

In [ ]:
cell_indices = np.arange(XV.n_obs)
subset_indices = np.random.choice(cell_indices, size= int(XV.n_obs // xv) + 1, replace=False)
print(len(cell_indices))
print(len(subset_indices))

In [ ]:
# Subset the AnnData object
XV_subset = XV[subset_indices, :]
XV_subset

In [ ]:
import anndata as ad
adata = ad.concat(
    [IX_subset, XII_subset, XV_subset], 
    label='batch', 
    keys=["IX", "XII", "XV"], 
    join='inner'
)

In [ ]:
adata.obs['batch_init'] =  adata.obs_names.str.split("-").str[-1]
adata.obs['batch_init']

In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
adata.write_h5ad(work_dir + 'IX_XII_XV_inner_join.h5ad')

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

In [ ]:
adata = adata[:, adata.var['highly_variable']]

In [ ]:
# save unscaled and integrated dataset for NMF analysis
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
adata.write_h5ad(work_dir + 'IX_XII_XV_inner_join_top2000.h5ad')

# 6 Scanpy preprocessing: scale, pca and harmony integration

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50)

In [ ]:
# Batch correction using harmony
sc.external.pp.harmony_integrate(adata, key = 'batch_init')

In [ ]:
sc.pl.embedding(adata, color='batch', basis='X_pca_harmony', palette=['green', 'darkblue', 'brown'],
               #save = '_batch.pdf'
               )

In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
adata.write_h5ad(work_dir + 'IX_XII_XV_top2000_harm.h5ad')

# 7 dimensionality reduction

In [ ]:
import scanpy as sc

work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
adata = sc.read_h5ad(work_dir + 'IX_XII_XV_top2000_harm.h5ad')

In [ ]:
# dimensionality reduction
#comps = [10, 15, 20, 25, 30, 35, 40, 45, 50]
comps = [25]
for comp in comps:
    neighbors_key = f'neighbors_{comp}dims_harm'  # Unique neighbors key
    umap_key = f'umap_{comp}dims_harm'           # Unique UMAP key for obsm
    
    sc.pp.neighbors(adata, n_pcs=comp, key_added=neighbors_key, use_rep='X_pca_harmony')
    sc.tl.umap(adata, neighbors_key=neighbors_key)
    adata.obsm[umap_key] = adata.obsm['X_umap']
    
    print(f"Computed UMAP for {comp} components. Stored in adata.obsm['{umap_key}']")

# 8 leiden clustering
here we performed over clustering of the dataset to identify clusters that are artifacts from low total count values, low gene counts, ribosomal genes etc.

In this study, we determined optimal clustering resolution using the silhouette score. Here for the over clustering it is not needed, however, later for comparing different clustering resolutions on different dimensionality reductions it will be helpful. 

In [ ]:
import pandas as pd
from sklearn.metrics import silhouette_score

# Define resolutions and dimensionality reductions to test
#resolutions = [0.08, 0.1, 0.15, 0.2, 1.0, 2.0, 5.0, 10.0]
#comps = [10, 15, 20, 25, 30, 35, 40, 45, 50]
resolutions = [2.0]
comps = [25]


# Corresponding reductions and neighbors already computed
reductions = [f'umap_{comp}dims_harm' for comp in comps]  # Keys in adata.obsm
neighbors_keys = [f'neighbors_{comp}dims_harm' for comp in comps]  # Keys in adata

# Initialize a list to store results
results = []

# Loop over each reduction and its corresponding neighbors_key
for reduction, neighbors_key in zip(reductions, neighbors_keys):
    print(f"\nEvaluating Silhouette Scores for Reduction: {reduction} with neighbors {neighbors_key}")
    
    # Loop over each resolution
    for res in resolutions:
        # Run Leiden clustering with the current resolution
        sc.tl.leiden(
            adata,
            resolution=res,
            random_state=0,
            n_iterations=2,
            directed=False,
            neighbors_key=neighbors_key,  # Use the correct neighbors_key
            key_added=f'leiden_{reduction}_{res}'  # Unique key for each reduction-resolution combo
        )
        
        # Retrieve cluster labels
        leiden_key = f'leiden_{reduction}_{res}'
        labels = adata.obs[leiden_key].astype('category').cat.codes

        # Get the embedding for the current reduction
        if reduction in adata.obsm:
            embedding = adata.obsm[reduction]
        else:
            print(f"Warning: {reduction} not found in adata.obsm. Skipping.")
            continue

        # Compute Silhouette Coefficient
        score = silhouette_score(embedding, labels)

        # Append the result as a row to the results list
        results.append({
            'Reduction': reduction,
            'NeighborsKey': neighbors_key,
            'Resolution': res,
            'SilhouetteScore': score
        })
        print(f"Reduction {reduction}, Resolution {res}: Silhouette Coefficient = {score:.3f}")

# Convert results into a pandas DataFrame
silhouette_df = pd.DataFrame(results)

# Display the DataFrame
print("\nSilhouette Coefficients DataFrame:")
print(silhouette_df)

In [ ]:
silhouette_df.loc[silhouette_df['SilhouetteScore'].idxmax()]

# 9 quality check for dataset

In [ ]:
import scanpy as sc
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
adata = sc.read_h5ad(work_dir + 'IX_XII_XV_top2000_harm.h5ad')

In [ ]:
basis = 'umap_25dims_harm'
resolution = f'leiden_{basis}_2.0'

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)

basis = 'umap_25dims_harm'
resolution = f'leiden_{basis}_2.0'

sc.pl.embedding(adata, color = resolution, basis=basis, save='_quality_check_resolution.pdf')

In [ ]:
sc.settings.set_figure_params(dpi=150, facecolor="white", color_map='inferno', frameon=False)


batch_of_interest = 'IX'
adata.obs['highlight_batch_IX'] = adata.obs['batch'].apply(lambda x: x if x == batch_of_interest else 'Other')
sc.pl.embedding(adata, color='highlight_batch_IX', basis=basis, palette=['green', 'lightgrey'], save='_quality_check_highligh_IX.pdf')

In [ ]:
sc.settings.set_figure_params(dpi=150, facecolor="white", color_map='inferno', frameon=False)


batch_of_interest = 'XII'
adata.obs['highlight_batch_XII'] = adata.obs['batch'].apply(lambda x: x if x == batch_of_interest else 'Other')
sc.pl.embedding(adata, color='highlight_batch_XII', basis=basis, palette=['lightgrey', 'darkblue'], save='_quality_check_highligh_XII.pdf')

In [ ]:
sc.settings.set_figure_params(dpi=150, facecolor="white", color_map='inferno', frameon=False)


batch_of_interest = 'XV'
adata.obs['highlight_batch_XV'] = adata.obs['batch'].apply(lambda x: x if x == batch_of_interest else 'Other')
sc.pl.embedding(adata, color='highlight_batch_XV', basis=basis, palette=['lightgrey', 'brown'], save='_quality_check_highligh_XV.pdf')

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno')
sc.pl.embedding(adata, color = ["n_genes_by_counts", "total_counts"], basis=basis, save='_quality_check_counts.pdf')

In [ ]:
# Create a boolean column for cells with < 1000 total counts
adata.obs['low_n_genes_by_counts'] = adata.obs['n_genes_by_counts'] < 1000

sc.pl.embedding(adata, color = ["low_n_genes_by_counts"], basis=basis, palette=['lightgrey', 'darkblue'])

In [ ]:
# Create a boolean column for cells with < 1000 total counts
adata.obs['low_total_counts'] = adata.obs['total_counts'] < 1000

sc.pl.embedding(adata, color = ["low_total_counts"], basis=basis, palette=['lightgrey', 'darkblue'])

In [ ]:
import itertools

sc.settings.set_figure_params(dpi=150, facecolor="white", color_map='plasma')
sc.pl.stacked_violin(adata, 'n_genes_by_counts', groupby=f'leiden_{basis}_{res}', swap_axes=True);

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='plasma', frameon=False)

adata.obs['low_n_genes_by_counts'] = adata.obs['n_genes_by_counts'] < 1000
basis = 'umap_25dims_harm'
resolution = f'leiden_{basis}_2.0'

sc.pl.stacked_violin(adata, 'low_n_genes_by_counts', groupby=resolution, swap_axes=True, save='_quality_check_low_genes.pdf'
                    );

In [ ]:
## ribosomoal genes

In [ ]:
import pandas as pd
import itertools
import numpy as np

df = pd.read_csv('/staging/leuven/stg_00171/Mark/interproscan_MP/ribo_OctVulgeneIDs.csv', header= None)
ribo_genes = df[0].tolist()
remove = ['OctVul6B008752', 'g.minus_peak_21598']
ribo_genes = [x for x in ribo_genes if x not in remove]

basis = 'umap_25dims_harm'
resolution = f'leiden_{basis}_2.0'

sc.settings.set_figure_params(dpi=300, facecolor="white")
sc.pl.stacked_violin(adata, ribo_genes, groupby=resolution, swap_axes=True, save='ribo_genes_quality.pdf');

In [ ]:
# save the cell counts for the clusters, that will be removed
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_check/'

cells_per_cluster = pd.DataFrame(adata.obs[resolution].value_counts())

cells_per_cluster.to_csv(work_dir + "cells_per_cluster.csv", index=False)
cells_per_cluster

In [ ]:
metrics = []
metrics = pd.DataFrame(metrics)
metrics['n_genes_by_counts'] = adata.obs['n_genes_by_counts']
metrics['low_n_genes_by_counts'] = adata.obs['low_n_genes_by_counts']
metrics['total_counts'] = adata.obs['total_counts']
metrics['low_total_counts'] = adata.obs['low_total_counts']

In [ ]:
# save the quality metrics for the cells
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_check/'

metrics.to_csv(work_dir + "n_genes_total_counts_allcells.csv", index=False)

# 10 subset for good quality clusters

In [ ]:
whole = adata

In [ ]:
whole.obs['batch'].value_counts()

In [ ]:
clusters_to_remove = ['15','20', # low counts
                      '22', # ribosomal
                      '24','26', # low counts
                      '28', # ribosomal
                      '37','38', # low counts
                      '41', # ribosomal
                      '42','43','44','45','46','47','48','49','50','51','52','53','54','55','56','57','58','59','60','61','62','63','64','65','66','67','68','69' # low counts
                     ]

In [ ]:
adata = whole[~whole.obs[resolution].isin(clusters_to_remove)]

In [ ]:
adata.obs['batch'].value_counts()

In [ ]:
len(adata)

# 11 adapt the number of cells from the stages to be the same again

In [ ]:
import numpy as np

# Your adata.obs['batch'] contains the batch labels, e.g. 'IX', 'XII', 'XV'
batch_col = 'batch'
min_cells = 21463
np.random.seed(33)

# Create a list of adata subsets
adata_balanced_batches = []

for batch, subadata in adata.obs.groupby(batch_col):
    if len(subadata) > min_cells:
        # Randomly sample barcodes
        sampled_barcodes = np.random.choice(subadata.index, size=min_cells, replace=False)
        adata_balanced_batches.append(adata[sampled_barcodes])
    else:
        # Keep as-is if already smallest
        adata_balanced_batches.append(adata[subadata.index])

# Concatenate back into one AnnData object
import scanpy as sc
adata_balanced = adata_balanced_batches[0].concatenate(
    *adata_balanced_batches[1:], 
    join='outer', 
    batch_key=batch_col
)

In [ ]:
# Optional: check counts again
print(adata_balanced.obs[batch_col].value_counts())

In [ ]:
adata = adata_balanced

# 12 scale, pca, harmony

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=100)

In [ ]:
# Batch correction using harmony
sc.external.pp.harmony_integrate(adata, key = 'batch_init')

In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/'
adata.write_h5ad(work_dir + 'IX_XII_XV_top2000_harm.h5ad')

# 13 umap dimensionality reduction, leiden clustering, silhouette score

In [ ]:
# ran dimensionality reduction and clustering analysis with silhouette score on 100 dims via job submission on cluster
# /staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/100dims
 

In [ ]:
import scanpy as sc
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/'
adata = sc.read_h5ad(work_dir + 'IX_XII_XV_top2000_harm.h5ad')

# dimensionality reduction
comps = [10, 15,  20, 25,  30, 35,  40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]

for comp in comps:
    neighbors_key = f'neighbors_{comp}dims_harm'  # Unique neighbors key
    umap_key = f'umap_{comp}dims_harm'           # Unique UMAP key for obsm

    sc.pp.neighbors(adata, n_pcs=comp, key_added=neighbors_key, use_rep='X_pca_harmony')
    sc.tl.umap(adata, neighbors_key=neighbors_key)
    adata.obsm[umap_key] = adata.obsm['X_umap']

    print(f"Computed UMAP for {comp} components. Stored in adata.obsm['{umap_key}']")

import pandas as pd
from sklearn.metrics import silhouette_score

# Define resolutions and dimensionality reductions to test
resolutions = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5, 2.0]

# Corresponding reductions and neighbors already computed
reductions = [f'umap_{comp}dims_harm' for comp in comps]  # Keys in adata.obsm
neighbors_keys = [f'neighbors_{comp}dims_harm' for comp in comps]  # Keys in adata

# Initialize a list to store results
results = []

# Loop over each reduction and its corresponding neighbors_key
for reduction, neighbors_key in zip(reductions, neighbors_keys):
    print(f"\nEvaluating Silhouette Scores for Reduction: {reduction} with neighbors {neighbors_key}")

    # Loop over each resolution
    for res in resolutions:
        # Run Leiden clustering with the current resolution
        sc.tl.leiden(
            adata,
            resolution=res,
            random_state=0,
            n_iterations=2,
            directed=False,
            neighbors_key=neighbors_key,  # Use the correct neighbors_key
            key_added=f'leiden_{reduction}_{res}'  # Unique key for each reduction-resolution combo
        )

        # Retrieve cluster labels
        leiden_key = f'leiden_{reduction}_{res}'
        labels = adata.obs[leiden_key].astype('category').cat.codes

        # Get the embedding for the current reduction
        if reduction in adata.obsm:
            embedding = adata.obsm[reduction]
        else:
            print(f"Warning: {reduction} not found in adata.obsm. Skipping.")
            continue

        # Compute Silhouette Coefficient
        score = silhouette_score(embedding, labels)

        # Append the result as a row to the results list
        results.append({
            'Reduction': reduction,
            'NeighborsKey': neighbors_key,
            'Resolution': res,
            'SilhouetteScore': score
        })
        print(f"Reduction {reduction}, Resolution {res}: Silhouette Coefficient = {score:.3f}")

# Convert results into a pandas DataFrame
silhouette_df = pd.DataFrame(results)

# Display the DataFrame
print("\nSilhouette Coefficients DataFrame:")
print(silhouette_df)

work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/100dims/'
adata.write_h5ad(work_dir + 'IX_XII_XV_top2000_harm_100dims.h5ad')
silhouette_df.to_csv(work_dir + "IX_XII_XV_top2000_harm_corase_silhouette_scores.csv", index=False)
                                                                                                                 

In [ ]:
import scanpy as sc
import pandas as pd

work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/100dims/'
adata = sc.read_h5ad(work_dir + 'IX_XII_XV_top2000_harm_100dims.h5ad')
silhouette_df = pd.read_csv(work_dir + "IX_XII_XV_top2000_harm_corase_silhouette_scores.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

heatmap_data = silhouette_df.pivot(index='Resolution', columns='Reduction', values='SilhouetteScore')

plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="coolwarm", cbar_kws={'label': 'SilhouetteScore'})
plt.title('SilhouetteScore Across Reductions and Resolutions', fontsize=14)
plt.xlabel('Dimensionality Reduction', fontsize=12)
plt.ylabel('Resolution', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figures/paper/Figure 1/silhouette_score_qualitycheck.pdf', dpi=300)
plt.show()


In [ ]:
silhouette_df.loc[silhouette_df['SilhouetteScore'].idxmax()]

In [ ]:
basis = 'umap_20dims_harm'
resolution = f'leiden_{basis}_0.1'
markers = f'{resolution}_markers'

In [ ]:
sc.settings.set_figure_params(dpi=150, facecolor="white", color_map='inferno', frameon=False)
sc.pl.embedding(adata, color = resolution, basis=basis)

# 14 annotate clusters 

In [ ]:
# read in orthology file to merge with octvul gene ids and use annotation for annotating the clusters
import pandas as pd
outdir = '/staging/leuven/stg_00171/Mark/OrthoFinder/2025_08_proteomes/primary_transcripts/OrthoFinder/Results_Sep29/orthologue_tables/'
ortho = pd.read_csv(outdir + 'LookUp_Ovul_MP_Dmel_Hsap.tsv', sep='\t')

In [ ]:
basis = 'umap_20dims_harm'
resolution = f'leiden_{basis}_0.1'
markers = f'{resolution}_markers'

result = adata.uns[markers]
groups = result["names"].dtype.names 
df = pd.DataFrame(
    {
        f"{group}_{key[:1]}": result[key][group]
        for group in groups
        for key in ["names", "pvals", "logfoldchanges"]
    }
).head(20)

In [ ]:
cluster = '0'
columns_to_extract = [f"{cluster}_n", f"{cluster}_p", f"{cluster}_l"]
new_df = df[columns_to_extract]
new_df.sort_values(by = f'{cluster}_l', ascending=False, inplace=True)
df_2 = new_df.merge(ortho, how='left', left_on=f'{cluster}_n', right_on='Ovul_MP_TD')
#df_2.to_csv(marker_dir + f'cluster_{cluster}')
df_2.head(20)

In [ ]:
basis = 'umap_20dims_harm'
resolution = f'leiden_{basis}_0.1'

adata.obs['coarse_anno'] = adata.obs[resolution].replace({'0': 'NDIFF', 
                                                         '1': 'MES', 
                                                         '2': 'EPI', 
                                                         '3': 'NPROG', 
                                                         '4': 'GLIA', 
                                                         '5': 'RET', 
                                                         '6': 'ENDOT'})

In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/100dims/'
adata.write_h5ad(work_dir + 'IX_XII_XV_top2000_harm_100dims.h5ad')

# 15 create an adata dataset with only the information of the coarse annotation
reduces size of the file

In [ ]:
import scanpy as sc
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/'
adata = sc.read_h5ad(work_dir + 'IX_XII_XV_top2000_harm.h5ad')

In [ ]:
# dimensionality reduction
comp = 20

neighbors_key = f'neighbors_{comp}dims_harm'  # Unique neighbors key
umap_key = f'umap_{comp}dims_harm'           # Unique UMAP key for obsm
    
sc.pp.neighbors(adata, n_pcs=comp, key_added=neighbors_key, use_rep='X_pca_harmony')
sc.tl.umap(adata, neighbors_key=neighbors_key)
adata.obsm[umap_key] = adata.obsm['X_umap']
  
print(f"Computed UMAP for {comp} components. Stored in adata.obsm['{umap_key}']")

In [ ]:
comp = 20
res = 0.1
reduction = f'umap_{comp}dims_harm'
neighbors_key = f'neighbors_{comp}dims_harm'

sc.tl.leiden(
            adata,
            resolution=res,
            random_state=0,
            n_iterations=2,
            directed=False,
            neighbors_key=neighbors_key,  # Use the correct neighbors_key
            key_added=f'leiden_{reduction}_{res}'  # Unique key for each reduction-resolution combo
)

In [ ]:
basis = 'umap_20dims_harm'
resolution = f'leiden_{basis}_0.1'

adata.obs['coarse_anno'] = adata.obs[resolution].replace({'0': 'NDIFF', 
                                                         '1': 'MES', 
                                                         '2': 'EPI', 
                                                         '3': 'NPROG', 
                                                         '4': 'GLIA', 
                                                         '5': 'RET', 
                                                         '6': 'ENDOT'})

In [ ]:
import scanpy as sc
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/'
adata.write_h5ad(work_dir + 'IX_XII_XV_top2000_harm_coarse_anno.h5ad')

# 16 create plots for figures

In [ ]:
import scanpy as sc
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/quality_subset/'
adata = sc.read_h5ad(work_dir + 'IX_XII_XV_top2000_harm_coarse_anno.h5ad')

In [ ]:
import itertools
import pandas as pd

resolution = 'coarse_anno'
#adata_red = adata[adata.obs[resolution].isin(['NPROG', 'NDIFF', 'RET', 'GLIA'])]

genes = [
    'OctVul6B022688', # SOX1/2/3
    'OctVul6B005118', # INSM1/2
    'OctVul6B013567', # NKX24/21
    'OctVul6B031766', # elav
    'OctVul6B031322', # onecut
    'OctVul6B030927', # Gli1/2/3
    'OctVul6B009174', # MEOX
    'OctVul6B018846', # ABCAC
    'OctVul6B005734', # grh
    'OctVul6B025607', # ASCL1
    'OctVul6B014263', # SNAI2
    'OctVul6B022728', # FOXN4
    'OctVul6B025188', # SP8
    'OctVul6B027541', # E2F
    'OctVul6B008109', # NR4A
    'OctVul6B029372', # DLX
    'OctVul6B028282', # apolpp
    'OctVul6B001883', # GFAP
    'OctVul6B003215', # OTX
    'OctVul6B010918', # MITF
    'OctVul6B000213', # opsin
    'OctVul6B022238', # NKX25
    'OctVul6B003406', # VGFR
]

sc.settings.set_figure_params(dpi=300, facecolor="white")
sc.settings.figdir = 'figures/paper/Figure 1/'

sc.pl.matrixplot(adata, var_names = genes, groupby ='coarse_anno', cmap='Reds', swap_axes=False, save='_matrix_plot_landscape.pdf');

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)

basis = 'umap_20dims_harm'
resolution = 'coarse_anno'

coarse_colors = ['#93C572', '#E3735E', '#235a5a', '#4a7c99', '#770737', '#F3CFC6', '#eeaf61'] 

sc.pl.embedding(adata, color = resolution, basis=basis, #save='_coarse_anno.pdf',
                palette=coarse_colors)

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)

batch_colors = {'IX':'#CC5500', 
                'XII': '#6082B6', 
                'XV': '#8A9A5B', 
                'Other': '#E5E4E2'
               }

highlights = ['highlight_batch_IX',
              'highlight_batch_XII',
              'highlight_batch_XV']

sc.pl.embedding(adata, color = highlights, basis=basis, save='_batch_highlights.pdf', 
                palette=batch_colors
               )

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Replace with your actual column names
cluster_col = 'coarse_anno'
batch_col = 'batch'

# Create a contingency table: number of cells per (cluster, batch)
contingency = pd.crosstab(adata.obs[cluster_col], adata.obs[batch_col])

# Normalize to proportions (per cluster)
proportions = contingency.div(contingency.sum(axis=1), axis=0)

# Plot
proportions.plot(kind='bar', stacked=False, figsize=(10,6), color=['#CC5500','#6082B6', '#8A9A5B'])
plt.ylabel('Proportion per cluster')
plt.title('Batch contribution per cluster')
plt.legend(title='Batch', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(False)
plt.savefig('figures/paper/Figure 1/scanpy_batch_contribution_coarse_anno.pdf', dpi=300)
plt.show()

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)
sc.settings.figdir = 'figures/paper/Figure 1/'

gene = ['pct_counts_mt', 
        'total_counts'
       ] 

basis = 'umap_20dims_harm'

sc.pl.embedding(adata, color=gene, basis=basis, #save='_selected_marker_genes.pdf',
                ncols=6)

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)
sc.settings.figdir = 'figures/paper/Figure 1/'

gene = ['OctVul6B025607', # ascl1
        'OctVul6B031322', # onecut
        'OctVul6B008109', # nr4a
        'OctVul6B028282', # apolpp
        'OctVul6B003215', # otx
        'OctVul6B000213', # opsin
       ] 

basis = 'umap_20dims_harm'

sc.pl.embedding(adata, color=gene, basis=basis, save='_selected_marker_genes.pdf', ncols=6)

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)

gene = ['OctVul6B025607', # ascl1
        'OctVul6B031766', # elav
        'OctVul6B015157', # grhl
        'OctVul6B003215', # otx
        'OctVul6B008109', # nr4a      
       ] 

basis = 'umap_20dims_harm'

sc.pl.embedding(adata, color=gene, basis=basis, save='_cluster_genes.pdf', ncols=5)

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)
sc.settings.figdir = 'figures/paper/Figure 2/'

gene = ['OctVul6B025607', # ascl1
        'OctVul6B013567', # nkx24/21
        'OctVul6B028298', # nkx61/62/63
        'OctVul6B026589', # nkx12      
       ] 

basis = 'umap_20dims_harm'

sc.pl.embedding(adata, color=gene, basis=basis, save='_ascl1_nkx24_nkx61_nkx12.pdf', 
                ncols=4)

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)
sc.settings.figdir = 'figures/plots/'

gene = ['OctVul6B013114', # gcm   
       ] 

basis = 'umap_20dims_harm'

sc.pl.embedding(adata, color=gene, basis=basis, save='_gcm.pdf', 
                ncols=5)

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)
sc.settings.figdir = 'figures/paper/Figure 6/HCR/'

gene = [
     'OctVul6B029372', # dlx   
        'OctVul6B026448', # tbx3/2
        
       ] 

basis = 'umap_20dims_harm'

sc.pl.embedding(adata, color=gene, basis=basis, save='_dlx_tbx2.pdf', 
                ncols=5)

In [ ]:
sc.settings.set_figure_params(dpi=300, facecolor="white", color_map='inferno', frameon=False)
sc.settings.figdir = 'figures/paper/Figure 6/'

basis = 'umap_20dims_harm'

cluster_of_interest = ['GLIA']
adata.obs['highlight_GLIA'] = adata.obs['coarse_anno'].isin(cluster_of_interest)
sc.pl.embedding(adata, color='highlight_GLIA', basis=basis, palette=['lightgrey', '#770737',], 
                save='_highligh_GLIA.pdf'
               )

# 17 add "coarse anno" annotation to unscaled object for NMF analysis

In [ ]:
adata.obs_names

In [ ]:
adata.obs_names = adata.obs_names.str.replace('-0', '')
adata.obs_names = adata.obs_names.str.replace('-1', '')
adata.obs_names = adata.obs_names.str.replace('-2', '')

In [ ]:
adata.obs_names

In [ ]:
adata.obs_names = adata.obs_names.str.replace('-IX', '-1-IX')
adata.obs_names = adata.obs_names.str.replace('-XII', '-1-XII')
adata.obs_names = adata.obs_names.str.replace('-XV', '-1-XV')

In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
unscaled = sc.read_h5ad(work_dir + 'IX_XII_XV_inner_join_top2000.h5ad')

In [ ]:
unscaled.obs_names

In [ ]:
unscaled.obs['coarse_anno'] = adata.obs['coarse_anno']

In [ ]:
work_dir = '/staging/leuven/stg_00171/Mark/IX_XII_XV_integration/mergedPeaks/all_datasets/'
unscaled.write_h5ad(work_dir + 'IX_XII_XV_inner_join_top2000_coarse_anno.h5ad')